# 🎬 Video Object Removal Pipeline
## SAM2 + ProPainter | Colab Version

**Requirements:** GPU runtime (T4 or better)

Run Cell 1 (setup), then Cell 2 (launch). You'll get a public URL with the full UI built in — no separate HTML file needed!

In [ ]:
# ============================================================
# 📦 CELL 1: SETUP — Install everything (run once)
# ============================================================
import os, sys

!git clone https://github.com/facebookresearch/sam2.git /content/sam2 -q
os.chdir('/content/sam2')
!pip install -e . -q
!wget -q -P /content/sam2/checkpoints/ https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt
!git clone https://github.com/sczhou/ProPainter.git /content/ProPainter -q
!pip install -r /content/ProPainter/requirements.txt -q
!pip install gradio -q

print("✅ ALL SETUP DONE!")

### 🚀 Launch the App
Run this cell to get a public URL with the complete UI.

In [ ]:
# ============================================================
# 🚀 CELL 2: LAUNCH APP — Full UI with video upload + object selection
# ============================================================

import gradio as gr
import os, sys, json, shutil, subprocess, tempfile
import torch
import numpy as np
import cv2

def extract_frame(video_path):
    """Extract first frame from video for object selection."""
    if not video_path:
        return None
    cap = cv2.VideoCapture(video_path)
    ret, frame = cap.read()
    cap.release()
    if ret:
        return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    return None

def remove_objects(video_path, coords_json):
    """Main pipeline: video + coordinates → result video."""
    try:
        if not video_path:
            return None, "❌ No video provided!"

        coords = json.loads(coords_json) if coords_json else []
        if not coords:
            return None, "❌ No coordinates provided! Click on the frame to select objects."

        work_dir = tempfile.mkdtemp(prefix="objremover_")
        frames_dir = os.path.join(work_dir, "frames")
        masks_dir = os.path.join(work_dir, "masks")
        output_dir = os.path.join(work_dir, "output")

        for d in [frames_dir, masks_dir, output_dir]:
            os.makedirs(d, exist_ok=True)

        # Extract frames
        subprocess.run([
            'ffmpeg', '-i', video_path, '-q:v', '2',
            os.path.join(frames_dir, '%05d.jpg'),
            '-hide_banner', '-loglevel', 'quiet'
        ], check=True)

        num_frames = len([f for f in os.listdir(frames_dir) if f.endswith('.jpg')])
        print(f"📁 Extracted {num_frames} frames")
        if num_frames == 0:
            return None, "❌ Could not extract frames!"

        # SAM2
        from hydra.core.global_hydra import GlobalHydra
        from hydra import initialize_config_dir
        from sam2.build_sam import build_sam2_video_predictor

        os.chdir('/content/sam2')
        if '/content/sam2' not in sys.path:
            sys.path.insert(0, '/content/sam2')
        GlobalHydra.instance().clear()

        with initialize_config_dir(config_dir="/content/sam2/sam2/configs/sam2.1", version_base=None):
            predictor = build_sam2_video_predictor(
                "sam2.1_hiera_l.yaml",
                "/content/sam2/checkpoints/sam2.1_hiera_large.pt",
                device="cuda"
            )

        inference_state = predictor.init_state(video_path=frames_dir)

        obj_id = 1
        for coord in coords:
            if coord.get('type') == 'point':
                predictor.add_new_points_or_box(
                    inference_state=inference_state, frame_idx=0, obj_id=obj_id,
                    points=np.array([[coord['x'], coord['y']]], dtype=np.float32),
                    labels=np.array([1], dtype=np.int32)
                )
                print(f"📍 Point ({coord['x']}, {coord['y']}) → obj {obj_id}")
            elif coord.get('type') == 'box':
                predictor.add_new_points_or_box(
                    inference_state=inference_state, frame_idx=0, obj_id=obj_id,
                    box=np.array([coord['x1'], coord['y1'], coord['x2'], coord['y2']], dtype=np.float32)
                )
                print(f"▭ Box ({coord['x1']},{coord['y1']})→({coord['x2']},{coord['y2']}) → obj {obj_id}")
            obj_id += 1

        video_segments = {}
        for frame_idx, object_ids, masks in predictor.propagate_in_video(inference_state):
            video_segments[frame_idx] = {
                oid: masks[i].cpu().numpy() for i, oid in enumerate(object_ids)
            }

        print(f"🔍 Tracked {len(video_segments)} frames")

        for frame_idx, segments in video_segments.items():
            combined = None
            for oid, mask_data in segments.items():
                binary = (mask_data[0] > 0).astype(np.uint8) * 255
                combined = binary if combined is None else np.maximum(combined, binary)
            cv2.imwrite(os.path.join(masks_dir, f"{frame_idx+1:05d}.png"), combined)

        # ProPainter
        os.chdir('/content/ProPainter')
        sample = cv2.imread(os.path.join(frames_dir, '00001.jpg'))
        h, w = sample.shape[:2]

        subprocess.run([
            'python', 'inference_propainter.py',
            '--video', frames_dir, '--mask', masks_dir,
            '--output', output_dir, '--height', str(h), '--width', str(w)
        ], check=True)

        # Find output
        inpaint_path = None
        for root, dirs, files in os.walk(output_dir):
            for f in files:
                if 'inpaint' in f and f.endswith('.mp4'):
                    inpaint_path = os.path.join(root, f)
                    break

        if not inpaint_path:
            return None, "❌ ProPainter produced no output!"

        final_path = os.path.join(tempfile.gettempdir(), "result_video.mp4")
        subprocess.run([
            'ffmpeg', '-i', inpaint_path,
            '-c:v', 'libx264', '-pix_fmt', 'yuv420p',
            final_path, '-y', '-hide_banner', '-loglevel', 'quiet'
        ], check=True)

        size_mb = os.path.getsize(final_path) / (1024*1024)
        msg = f"✅ Removed {len(coords)} object(s) across {num_frames} frames ({size_mb:.1f} MB)"
        print(msg)
        return final_path, msg

    except Exception as e:
        import traceback
        traceback.print_exc()
        return None, f"❌ Error: {str(e)}"


# ─── BUILD GRADIO UI ───
with gr.Blocks(title="Video Object Remover", theme=gr.themes.Base(primary_hue="emerald")) as demo:
    gr.Markdown("# 🎬 Video Object Remover")
    gr.Markdown("Upload a video → click on the frame to select objects → hit Remove!")

    with gr.Row():
        with gr.Column(scale=1):
            video_input = gr.Video(label="📁 Upload Video")
            frame_preview = gr.Image(label="👆 Click on the object to remove", interactive=True, type="numpy")
            coords_display = gr.Textbox(label="📍 Coordinates (auto-filled when you click)", lines=3, interactive=True)

        with gr.Column(scale=1):
            result_video = gr.Video(label="🎬 Result Video")
            status_text = gr.Textbox(label="Status", interactive=False)

    remove_btn = gr.Button("🗑 Remove Objects", variant="primary", size="lg")

    # State to track coordinates
    coord_state = gr.State([])

    def on_video_upload(video):
        frame = extract_frame(video)
        return frame, "[]", []

    def on_frame_click(frame, evt: gr.SelectData, current_coords):
        if frame is None:
            return frame, "[]", []

        x, y = evt.index[0], evt.index[1]
        coord = {"type": "point", "x": int(x), "y": int(y), "frame_time": 0.0}

        if current_coords is None:
            current_coords = []
        current_coords.append(coord)

        # Draw marker on frame
        marked_frame = frame.copy()
        cv2.circle(marked_frame, (x, y), 8, (0, 255, 100), 2)
        cv2.circle(marked_frame, (x, y), 3, (0, 255, 100), -1)

        # Draw all previous markers too
        for c in current_coords[:-1]:
            cx, cy = c['x'], c['y']
            cv2.circle(marked_frame, (cx, cy), 8, (0, 255, 100), 2)
            cv2.circle(marked_frame, (cx, cy), 3, (0, 255, 100), -1)

        return marked_frame, json.dumps(current_coords, indent=2), current_coords

    def on_remove(video, coords_json):
        return remove_objects(video, coords_json)

    video_input.change(on_video_upload, [video_input], [frame_preview, coords_display, coord_state])
    frame_preview.select(on_frame_click, [frame_preview, coord_state], [frame_preview, coords_display, coord_state])
    remove_btn.click(on_remove, [video_input, coords_display], [result_video, status_text])

demo.launch(share=True, debug=True, allowed_paths=["/tmp", "/content"])